# Лекция 3. Функции потерь, метрики и оптимизация — код

Код из лекции: функции потерь для регрессии и классификации, метрики качества, выбор порога, оптимизаторы и переобучение.

Ячейки выполняются **по порядку**: каждая следующая использует переменные из предыдущих. Проще всего — «Среда выполнения → Выполнить все».

В Colab всё нужное уже установлено. Для раздела 3.3 нужен интернет: датасет German Credit скачивается с openml.org. На своём компьютере установите библиотеки командой `pip install torch scikit-learn pandas`.

## Глава 2. Функции потерь

### 2.1. Регрессия: MSE, MAE и Huber

Пять прогнозов цены квартиры (в млн ₽), все ошибки не больше 1. Потом портим один прогноз — промах на 10 — и сравниваем три потери: MSE, MAE и Huber.

In [ ]:
import torch
from torch import nn

# пять примеров: правильные ответы и прогнозы модели (например, цены квартир в млн ₽)
y_true = torch.tensor([3.0, 5.0, 2.0, 7.0, 4.0])     # правильные ответы
y_pred = torch.tensor([2.5, 5.5, 2.0, 6.0, 4.5])     # хороший прогноз: ошибки не больше 1
y_bad = y_pred.clone()                                # копия, чтобы не испортить y_pred
y_bad[3] = 17.0                                       # тот же прогноз, но с одним грубым промахом

# три функции потерь из PyTorch:
#   MSELoss   — средний квадрат ошибки
#   L1Loss    — средний модуль ошибки (MAE)
#   HuberLoss — квадрат для маленьких ошибок (до delta), модуль для больших
for name, loss_fn in [("MSE", nn.MSELoss()), ("MAE", nn.L1Loss()), ("Huber", nn.HuberLoss(delta=1.0))]:
    # loss_fn(прогноз, ответ) возвращает тензор из одного числа; .item() превращает его в число Python
    print(f"{name:5s}: без промаха {loss_fn(y_pred, y_true).item():.3f} | с промахом {loss_fn(y_bad, y_true).item():.3f}")

**Что получилось.** Один промах увеличил MSE почти в 60 раз (0.35 → 20.15), а MAE и Huber — примерно в 5 и 11 раз.

Важнее величины потери то, **куда её градиент поведёт модель**. Посмотрим градиенты по каждому прогнозу.

In [ ]:
for name, loss_fn in [("MSE", nn.MSELoss()), ("MAE", nn.L1Loss()), ("Huber", nn.HuberLoss(delta=1.0))]:
    # requires_grad_(True): просим PyTorch считать градиенты по прогнозам,
    # как будто прогнозы — это параметры, которые мы будем менять
    p = y_bad.clone().requires_grad_(True)
    loss_fn(p, y_true).backward()               # обратный проход: градиенты попадут в p.grad
    # p.grad — насколько сильно потеря «тянет» каждый прогноз; округляем для читаемости
    print(f"{name:5s}: градиенты по прогнозам {[round(g, 2) for g in p.grad.tolist()]}")

**Что получилось.** У MSE градиент по промаху (4.0) в 20 раз больше остальных (0.2): при обучении почти весь шаг уйдёт на один выброс. У MAE и Huber вклад промаха не больше, чем у обычной ошибки: производная $e^2$ равна $2e$, а у $|e|$ — всего $\pm 1$.

### 2.2. Бинарная классификация: BCEWithLogitsLoss

Два способа посчитать бинарную кросс-энтропию: сигмоида + `BCELoss` (способ 1) и сразу по логиту — `BCEWithLogitsLoss` (способ 2). Модель ошибается на обоих заёмщиках; сравним умеренные и большие логиты.

In [ ]:
print(torch.sigmoid(torch.tensor(30.0)).item())     # σ(30) в float32

y = torch.tensor([0.0, 1.0])        # заёмщик A вернул кредит (0), заёмщик B — не вернул (1)
for size in [3.0, 30.0]:            # умеренно уверенная модель и очень уверенная
    # логиты модели: про A — «не вернёт», про B — «вернёт»; оба раза модель ошибается
    z1 = torch.tensor([size, -size], requires_grad=True)
    # способ 1: сначала сигмоида превращает логиты в вероятности, затем BCELoss
    loss1 = nn.BCELoss()(torch.sigmoid(z1), y)
    loss1.backward()                                    # градиенты по логитам — в z1.grad
    # способ 2: те же логиты, но сигмоида спрятана внутри функции потерь
    z2 = torch.tensor([size, -size], requires_grad=True)
    loss2 = nn.BCEWithLogitsLoss()(z2, y)
    loss2.backward()                                    # градиенты по логитам — в z2.grad
    print(f"логиты ±{size:g}: способ 1 — потеря {loss1.item():6.2f}, градиенты {[round(g, 3) for g in z1.grad.tolist()]}")
    print(f"             способ 2 — потеря {loss2.item():6.2f}, градиенты {[round(g, 3) for g in z2.grad.tolist()]}")

* Первая строка: $\sigma(30)$ в `float32` округлилась ровно до 1.0.
* Логиты $\pm 3$ — оба способа совпадают.
* Логиты $\pm 30$ — способ 1 даёт потерю 65 вместо правильных 30, а градиент по заёмщику A — ноль: модель не узнаёт об ошибке. Способ 2 считает правильно.

Правило: последний слой выдаёт логиты, их передают в `BCEWithLogitsLoss`.

### 2.3. Многоклассовая классификация: CrossEntropyLoss и log-sum-exp

Softmax и большие логиты. Три сорта вина, модель уверена в сорте 0, а на самом деле это сорт 1.

In [ ]:
# 1. Сдвиг всех логитов на одно и то же число не меняет softmax
#    (dim=0: вероятности считаются вдоль единственной оси тензора)
print(torch.softmax(torch.tensor([2.0, 1.0, 0.0]), dim=0))
print(torch.softmax(torch.tensor([12.0, 11.0, 10.0]), dim=0))   # те же логиты + 10

# 2. Большие логиты: один пример (батч из одного вина), модель уверена в сорте 0
logits = torch.tensor([[1000.0, 0.0, -1000.0]])           # форма (1, 3): 1 пример, 3 класса
target = torch.tensor([1])                                # а правильный ответ — сорт 1
naive = torch.exp(logits) / torch.exp(logits).sum()       # softmax «по учебнику»: e^1000 = inf
print(naive)
print(-torch.log(naive[0, 1]))                            # потеря «по учебнику»: −log p(сорт 1)

# 3. Устойчивые версии из PyTorch: внутри используют вычитание максимума (log-sum-exp)
print(torch.log_softmax(logits, dim=1))                   # dim=1 — вдоль классов
print(nn.CrossEntropyLoss()(logits, target).item())       # принимает логиты, а не вероятности

1. Логиты $(2, 1, 0)$ и $(12, 11, 10)$ дают одинаковые вероятности: сдвиг всех логитов не меняет softmax.
2. Softmax «по учебнику» на логитах $(1000, 0, -1000)$ ломается: получаются `nan` и `inf`.
3. После вычитания максимума логиты $(0, -1000, -2000)$ — переполняться нечему.
4. `CrossEntropyLoss` делает это сама и выдаёт правильную потерю 1000.

### 2.4. Multi-label: сигмоида на каждой метке

Multi-label: два фильма и три жанра — комедия, ужасы, драма. У каждого жанра своя сигмоида и своя бинарная задача.

In [ ]:
# два фильма, у каждого три метки-жанра: комедия, ужасы, драма
# строка — фильм, столбец — жанр; в logits — «сырые» выходы модели
logits = torch.tensor([[ 2.0, -1.0,  0.5],     # фильм 1
                       [-1.5, -2.0, -0.5]])    # фильм 2
# правильные ответы: 1 — жанр есть, 0 — нет (дробные числа, как требует BCE)
labels = torch.tensor([[1.0, 0.0, 1.0],        # фильм 1: комедия и драма
                       [0.0, 0.0, 0.0]])       # фильм 2: ни один из трёх жанров
print(labels.shape, labels.dtype)              # та же форма, что у logits

probs = torch.sigmoid(logits)                  # у каждого жанра своя вероятность
print(probs)
# сравнение даёт True/False; .int() превращает их в 1/0
print((probs > 0.5).int())                     # ответ модели: порог 0.5 для каждого жанра отдельно

# reduction="none" — не усреднять, вернуть потерю на каждой метке (таблица 2×3)
print(nn.BCEWithLogitsLoss(reduction="none")(logits, labels))
print(round(nn.BCEWithLogitsLoss()(logits, labels).item(), 4))  # по умолчанию: среднее всех 6 чисел

* Правильные ответы — форма `(2, 3)`, тип `float32`; у второго фильма все нули — для multi-label это нормально.
* Вероятности фильма 1 — 0.88, 0.27, 0.62: в сумме не 1, и так и должно быть.
* С порогом 0.5: фильм 1 — «комедия и драма», фильм 2 — «ни один жанр».
* Потери — таблица 2 × 3, итоговая потеря — среднее всех шести чисел, 0.286.

Почему не softmax: посмотрим, что он дал бы для фильма 1 на тех же логитах.

In [ ]:
# logits[0] — первая строка, то есть логиты фильма 1
print(torch.softmax(logits[0], dim=0))         # что дал бы softmax для фильма 1

**Что получилось.** Драма упала до 0.18 и с порогом 0.5 потерялась: softmax «делит» единицу между жанрами. Сказать «ни один жанр» он не может. Для multi-label — только сигмоида на каждом выходе и `BCEWithLogitsLoss`.

### 2.5. Как потери складываются в одно число: reduction и веса классов

Параметр `reduction` — как потери отдельных примеров складываются в одно число. Батч из 4 примеров: три класса 0, один класса 1; модель всем выдаёт логиты $(2, 0)$ — «скорее класс 0».

In [ ]:
# 4 примера, 2 класса; про каждый пример модель выдаёт логиты (2, 0) — «скорее класс 0»
logits = torch.tensor([[2.0, 0.0], [2.0, 0.0], [2.0, 0.0], [2.0, 0.0]])
target = torch.tensor([0, 0, 0, 1])                  # правильные ответы: последний пример — класс 1

# параметр reduction говорит, как свести потери отдельных примеров в одно число
print(nn.CrossEntropyLoss(reduction="none")(logits, target))                 # потеря каждого примера
print(round(nn.CrossEntropyLoss(reduction="sum")(logits, target).item(), 4))   # сумма
print(round(nn.CrossEntropyLoss(reduction="mean")(logits, target).item(), 4))  # среднее (по умолчанию)

* `"none"` — потеря каждого примера: $-\ln 0.881 = 0.127$ для трёх правильных и $-\ln 0.119 = 2.127$ для ошибки.
* `"sum"` — сумма 2.508. Она растёт с размером батча.
* `"mean"` (по умолчанию) — $2.508 / 4 = 0.627$, от размера батча не зависит.

Веса классов: дадим редкому классу 1 вес 3.

In [ ]:
w = torch.tensor([1.0, 3.0])                         # вес класса 0 — 1, вес класса 1 — 3
# потеря каждого примера умножается на вес его правильного класса
print(nn.CrossEntropyLoss(weight=w, reduction="none")(logits, target))
# среднее с весами: сумма взвешенных потерь делится на сумму весов (1+1+1+3 = 6), а не на 4
print(round(nn.CrossEntropyLoss(weight=w)(logits, target).item(), 4))

**Что получилось.** Потеря четвёртого примера умножилась на 3 и стала 6.381. Среднее делится на сумму весов: $(3 \cdot 0.127 + 6.381) / 6 \approx 1.127$ — как будто пример класса 1 лежал в батче трижды.

### 2.6. Label smoothing: не позволяем модели быть слишком уверенной

Label smoothing: модель всё увереннее выбирает правильный класс 0 (логиты $(\text{gap}, 0, 0)$). Сравним потерю без сглаживания и с `label_smoothing=0.1`.

In [ ]:
target = torch.tensor([0])                            # правильный класс — 0 (из трёх)
print(" разница | вероятность |  потеря без  |  потеря со")
print(" логитов |   класса 0  | сглаживания  | сглаживанием")
for gap in [0.0, 1.0, 2.0, 3.33, 5.0, 10.0, 20.0]:
    logits = torch.tensor([[gap, 0.0, 0.0]])          # чем больше gap, тем увереннее модель в классе 0
    p = torch.softmax(logits, dim=1)[0, 0].item()     # вероятность класса 0 у первого (единственного) примера
    plain = nn.CrossEntropyLoss()(logits, target).item()
    # label_smoothing=0.1: цель не (1, 0, 0), а (0.933, 0.033, 0.033)
    smooth = nn.CrossEntropyLoss(label_smoothing=0.1)(logits, target).item()
    print(f"{gap:7g}  | {p:10.4f}  | {plain:10.4f}   | {smooth:10.4f}")

* Без сглаживания потеря только убывает: модели всегда выгодно раздуть логит ещё сильнее.
* Со сглаживанием у потери есть минимум — 0.29 при вероятности 0.933, то есть на мягкой цели. Дальше уверенность наказывается: при вероятности 0.9999 потеря уже 0.67.

## Глава 3. Метрики качества

### 3.2. Метрики регрессии

Метрики регрессии на тех же пяти прогнозах цены.

In [ ]:
# y_true и y_pred — пять цен и прогноз из раздела 2.1 (в млн ₽)
errors = y_pred - y_true                       # ошибка на каждом примере (в млн ₽)
print("ошибки:", errors.tolist())              # .tolist() — тензор → обычный список Python

mse = (errors ** 2).mean()                     # средний квадрат ошибки (в «квадратных миллионах»)
rmse = mse.sqrt()                              # корень из MSE — снова в млн ₽
mae = errors.abs().mean()                      # средний модуль ошибки (в млн ₽)
print(f"MSE = {mse:.3f}, RMSE = {rmse:.3f}, MAE = {mae:.3f}")

**Что получилось.** MAE = 0.5 — в среднем ошибаемся на полмиллиона рублей. MSE = 0.35 — в «квадратных миллионах рублей». RMSE = 0.59 — снова в млн ₽; она больше MAE, потому что крупные ошибки весят сильнее.

Тот же прогноз, но с одним грубым промахом на 10 млн.

In [ ]:
e_bad = y_bad - y_true                         # прогноз с одним грубым промахом из раздела 2.1
print("ошибки:", e_bad.tolist())
# RMSE возводит ошибки в квадрат, поэтому один промах раздувает её сильнее, чем MAE
print(f"RMSE = {(e_bad ** 2).mean().sqrt():.3f}, MAE = {e_bad.abs().mean():.3f}")

**Что получилось.** MAE выросла в 4.6 раза, RMSE — в 7.6. Если RMSE заметно больше MAE, среди ошибок есть крупные.

$R^2$ — насколько модель лучше «глупого» прогноза, который всегда отвечает среднюю цену.

In [ ]:
baseline = y_true.mean()                       # «глупый» прогноз: всегда средняя цена
sse_model = (errors ** 2).sum()                # сумма квадратов ошибок модели
sse_base = ((y_true - baseline) ** 2).sum()    # сумма квадратов ошибок «глупого» прогноза
r2 = 1 - sse_model / sse_base                  # 1 — идеально, 0 — не лучше «глупого» прогноза
print(f"средняя цена: {baseline:.1f}")
print(f"ошибка модели: {sse_model:.2f}, ошибка «глупого» прогноза: {sse_base:.2f}")
print(f"R² = {r2:.3f}")

**Что получилось.** Ошибка модели в 8.5 раз меньше, чем у «глупого» прогноза: $R^2 = 1 - 1.75 / 14.8 \approx 0.88$. $R^2 = 1$ — идеал, 0 — не лучше среднего, меньше 0 — хуже среднего.

### 3.3. Матрица ошибок

Кредитный скоринг на датасете German Credit: 1000 заёмщиков немецкого банка. **Нужен интернет** — `fetch_openml` скачивает данные с openml.org. Обучаем один нейрон — логистическую регрессию.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Данные: 1000 заёмщиков немецкого банка, 20 признаков (сумма, срок, цель кредита...)
credit = fetch_openml("credit-g", version=1, as_frame=True)     # скачивается с openml.org
# убираем столбцы, которые нельзя использовать для решения о кредите: пол и гражданство
features = credit.data.drop(columns=["personal_status", "foreign_worker"])
# get_dummies: категория («цель: машина», «цель: ремонт», ...) → столбцы из нулей и единиц
X = pd.get_dummies(features).astype(float).to_numpy()
y = (credit.target == "bad").astype(int).to_numpy()     # 1 — не вернул кредит, 0 — вернул
print(X.shape, np.bincount(y))                           # bincount: сколько нулей и сколько единиц

# 2. Делим на обучение (75%) и валидацию (25%);
#    stratify=y — доля невозвратов в обеих частях одинаковая
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
# стандартизация, как в лекции 2: среднее и разброс считаем только по обучающим данным
scaler = StandardScaler().fit(X_train)
# переводим всё в тензоры PyTorch; BCE требует дробные ответы, поэтому y тоже float32
X_train_t = torch.tensor(scaler.transform(X_train), dtype=torch.float32)
X_val_t = torch.tensor(scaler.transform(X_val), dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# 3. Модель и обучение
torch.manual_seed(0)                         # одинаковые начальные веса при каждом запуске
model = nn.Linear(X.shape[1], 1)             # один нейрон — логистическая регрессия
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()           # модель выдаёт логит, сигмоида — внутри потери
for epoch in range(200):                     # данных немного — учим на всей выборке сразу
    optimizer.zero_grad()                    # обнуляем старые градиенты
    # model(...) имеет форму (750, 1); squeeze(1) убирает лишнюю ось → (750,), как у ответов
    loss = criterion(model(X_train_t).squeeze(1), y_train_t)
    loss.backward()                          # считаем градиенты
    optimizer.step()                         # делаем шаг оптимизатора

# 4. Прогноз на валидации: no_grad — градиенты здесь не нужны
with torch.no_grad():
    p_val = torch.sigmoid(model(X_val_t).squeeze(1)).numpy()   # вероятность невозврата
print(len(y_val), np.bincount(y_val))        # размер валидации и сколько в ней каждого класса

* 1000 заёмщиков; после one-hot кодирования (`get_dummies`) — 55 столбцов. 700 вернули кредит, 300 — нет.
* Пол и гражданство убраны: отказ по ним — дискриминация.
* На валидации 250 заёмщиков, из них 75 не вернули. `p_val` — вероятности невозврата, они понадобятся в следующих ячейках.

### 3.4. Accuracy, precision, recall и F1

Матрица ошибок и метрики при пороге 0.5. Положительный класс — «не вернёт кредит».

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

pred = (p_val >= 0.5).astype(int)            # порог 0.5: вероятность ≥ 0.5 → «не вернёт» (1)
# confusion_matrix возвращает таблицу 2×2, ravel() вытягивает её в 4 числа по порядку
tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"accuracy  = {accuracy_score(y_val, pred):.3f}")     # доля верных ответов
print(f"precision = {precision_score(y_val, pred):.3f}")    # из отказов — сколько справедливых
print(f"recall    = {recall_score(y_val, pred):.3f}")       # из невозвратов — сколько поймали
print(f"F1        = {f1_score(y_val, pred):.3f}")           # среднее гармоническое двух последних
# для сравнения: «модель», которая всем говорит 0 (выдать кредит)
print(f"«всем выдать кредит»: accuracy = {accuracy_score(y_val, np.zeros_like(y_val)):.3f}")

* Accuracy 0.772 — а стратегия «всем выдать» получает 0.700, не найдя ни одного риска.
* Precision 0.641 — из 64 отказов 41 справедливый, 23 надёжным заёмщикам отказали зря.
* Recall 0.547 — нашли 41 невозврат из 75, пропустили 34.
* F1 0.590 — низкое, если низко хоть одно из двух.

### 3.5. Порог решает, какой ошибки будет меньше

Порог решает, какой ошибки будет меньше. Переберём несколько порогов; стоимость считаем как $5 \cdot FN + 1 \cdot FP$ — невозврат в 5 раз дороже напрасного отказа.

In [ ]:
for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]:
    pred = (p_val >= thr).astype(int)                       # 1 — «не вернёт», отказ; 0 — выдать кредит
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()  # матрица ошибок из раздела 3.3
    cost = 5 * fn + 1 * fp              # невозврат в 5 раз дороже, чем упущенный клиент
    print(f"порог {thr}: FP={fp:3d} FN={fn:2d} | precision {precision_score(y_val, pred):.3f} "
          f"recall {recall_score(y_val, pred):.3f} F1 {f1_score(y_val, pred):.3f} | стоимость {cost}")
print("«всем выдать кредит»: стоимость", 5 * y_val.sum())   # без модели: все 75 невозвратов — потери

**Что получилось.** Ниже порог — меньше пропусков (FN), больше напрасных отказов (FP). Минимальная стоимость — 129 при пороге 0.3, против 193 при пороге 0.5 и 375 у «всем выдать». Максимум F1 — при 0.4: F1 считает ошибки равноценными. Порог подбирают на валидации по цене ошибок.

### 3.6. Метрики, не зависящие от порога: ROC-AUC, PR-AUC, Brier score

Метрики, которые не зависят от порога.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# все три метрики получают вероятности p_val, а не ответы после порога
print(f"ROC-AUC = {roc_auc_score(y_val, p_val):.4f}")       # 0.5 — угадывание, 1 — идеал
print(f"PR-AUC (average precision) = {average_precision_score(y_val, p_val):.4f}")
print(f"Brier score = {brier_score_loss(y_val, p_val):.4f}")  # средний квадрат (p − y); меньше — лучше

**Что получилось.** ROC-AUC 0.80: в 80% пар «не вернул / вернул» риск первого оценён выше. PR-AUC (0.63) ниже — так обычно бывает при дисбалансе классов. Brier score 0.160 проверяет калибровку вероятностей; у прогноза «всем 0.3» он около 0.21.

### 3.7. Многоклассовые задачи: как усреднять

Многоклассовая задача: 10 обращений клиентов в банк, три класса — консультация, жалоба, мошенничество. Модель ошиблась один раз: жалобу приняла за консультацию.

In [ ]:
# 10 обращений клиентов в банк, три класса:
# 0 — консультация (частый класс), 1 — жалоба и 2 — мошенничество (редкие классы)
y_true_m = [0, 0, 0, 0, 0, 0, 0, 0, 1, 2]     # правильные ответы
y_pred_m = [0, 0, 0, 0, 0, 0, 0, 0, 0, 2]     # ответы модели: жалобу она приняла за консультацию

print(confusion_matrix(y_true_m, y_pred_m))   # строки — правильный класс, столбцы — ответ модели

# average=None — не усреднять, а вернуть метрику для каждого класса
# zero_division=0 — если делить не на что (класс ни разу не предсказан), считать метрику нулём
kw = dict(average=None, zero_division=0)
# **kw передаёт оба параметра из словаря, чтобы не повторять их три раза
print("precision по классам:", precision_score(y_true_m, y_pred_m, **kw).round(3))
print("recall    по классам:", recall_score(y_true_m, y_pred_m, **kw).round(3))
print("F1        по классам:", f1_score(y_true_m, y_pred_m, **kw).round(3))

**Что получилось.** Строка матрицы — правильный класс, столбец — ответ модели. Жалобы модель не распознаёт совсем: у класса 1 precision, recall и F1 — нули.

Три способа свести метрики классов в одно число.

In [ ]:
# micro    — считаем TP, FP, FN по всем классам вместе;
# macro    — простое среднее трёх F1, все классы равны;
# weighted — среднее трёх F1 с весами по числу примеров класса
for avg in ["micro", "macro", "weighted"]:
    print(f"F1 {avg:8s} = {f1_score(y_true_m, y_pred_m, average=avg, zero_division=0):.3f}")

* macro — простое среднее: $(0.941 + 0 + 1) / 3 = 0.647$; важны все классы, в том числе редкие.
* weighted — с весами 8, 1, 1: 0.853; провал на жалобах заметен слабее.
* micro — по суммам TP, FP, FN: 0.9, то же, что accuracy.

Прежде чем усреднять, посмотрите на метрики по каждому классу.

## Глава 4. Оптимизаторы

### 4.2. SGD с моментом

Оптимизаторы на «овраге» $f = w_1^2 + 50\,w_2^2$: обычный SGD и SGD с моментом, шаг $\eta = 0.019$.

In [ ]:
def descend(make_optimizer, steps=150):
    """Спускается по «оврагу» из точки (3, 4) и возвращает значение f на каждом шаге."""
    w = torch.tensor([3.0, 4.0], requires_grad=True)   # две координаты, которые оптимизируем
    opt = make_optimizer([w])                          # оптимизатор получает список параметров
    history = []
    for _ in range(steps):
        opt.zero_grad()
        f = w[0] ** 2 + 50 * w[1] ** 2        # «овраг»: вдоль w2 склон в 50 раз круче
        history.append(f.item())              # запоминаем высоту до шага
        f.backward()
        opt.step()
    return history

# lambda p: ... — короткая функция «получи параметры p, создай оптимизатор»;
# так descend может сам создать оптимизатор для своего w
for name, make in [("SGD", lambda p: torch.optim.SGD(p, lr=0.019)),
                   ("SGD + момент 0.7", lambda p: torch.optim.SGD(p, lr=0.019, momentum=0.7)),
                   ("SGD + момент 0.9", lambda p: torch.optim.SGD(p, lr=0.019, momentum=0.9))]:
    h = descend(make)
    # номер первого шага, на котором f < 0.01 (None, если так и не спустились)
    first = next((i for i, v in enumerate(h) if v < 0.01), None)
    print(f"{name:17s}: f после 60 шагов = {h[60]:.2e}, f < 0.01 впервые на шаге {first}")

**Что получилось.** Момент 0.7 ускорил спуск почти втрое: $f < 0.01$ за 30 шагов вместо 88. Момент 0.9 при том же шаге хуже обычного SGD: инерция слишком велика, «шар» перелетает дно. Момент и скорость обучения подбирают вместе.

### 4.5. Сравним на Wine

Сеть из лекции 2 на Wine и четыре оптимизатора — с одинаковыми начальными весами и одинаковым порядком батчей.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_wine

# данные из лекции 2: 178 вин, 13 химических признаков, 3 сорта
Xw, yw = load_wine(return_X_y=True)
Xw_tr, Xw_val, yw_tr, yw_val = train_test_split(Xw, yw, test_size=0.2, random_state=42, stratify=yw)
sc_w = StandardScaler().fit(Xw_tr)            # стандартизация по обучающей части
# TensorDataset склеивает признаки и ответы в пары (x, y), чтобы DataLoader выдавал батчи
wine_train = TensorDataset(torch.tensor(sc_w.transform(Xw_tr), dtype=torch.float32), torch.tensor(yw_tr))
Xw_val_t = torch.tensor(sc_w.transform(Xw_val), dtype=torch.float32)
yw_val_t = torch.tensor(yw_val)

# четыре оптимизатора; у каждого свой привычный шаг обучения lr
optimizers = {
    "SGD":            lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD + momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "RMSprop":        lambda p: torch.optim.RMSprop(p, lr=1e-3),
    "Adam":           lambda p: torch.optim.Adam(p, lr=1e-3),
}
for name, make in optimizers.items():
    torch.manual_seed(42)                                 # одинаковые начальные веса для всех
    net = nn.Sequential(nn.Linear(13, 32), nn.ReLU(), nn.Linear(32, 3))   # сеть из лекции 2
    opt = make(net.parameters())
    loader = DataLoader(wine_train, batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(42))   # одинаковый порядок батчей
    ce = nn.CrossEntropyLoss()
    val_losses = []                                       # потеря на валидации после каждой эпохи
    for epoch in range(30):
        for xb, yb in loader:                             # обычный цикл обучения по батчам
            opt.zero_grad()
            ce(net(xb), yb).backward()
            opt.step()
        with torch.no_grad():                             # проверка на валидации без градиентов
            val_losses.append(ce(net(Xw_val_t), yw_val_t).item())
    # эпохи нумеруем с 1, а список — с 0, поэтому эпоха 10 — это val_losses[9]
    print(f"{name:15s}: val loss после эпох 1/10/20/30 = "
          f"{val_losses[0]:.3f} / {val_losses[9]:.3f} / {val_losses[19]:.3f} / {val_losses[29]:.3f}")

* Все четыре стартуют около $\ln 3 \approx 1.1$ — потери необученной модели.
* Обычный SGD с шагом 0.01 медленнее всех: 0.47 после 30 эпох. Момент при том же шаге — 0.065.
* RMSprop и Adam с шагом 0.001 тоже сходятся благодаря подстройке шага под каждый вес.

Оптимизатора, который всегда лучше, не существует: результат зависит от задачи и скорости обучения.

## Глава 5. Переобучение и регуляризация

### 5.1. Как выглядит переобучение

Переобучение: большая сеть (два скрытых слоя по 256 нейронов, около 80 тысяч параметров) на кредитных данных, 300 эпох. Сравним вариант без регуляризации и вариант с dropout и weight decay. Ячейка может работать около минуты.

In [ ]:
def train_big(dropout, weight_decay, epochs=300):
    """Учит большую сеть на данных о кредитах; возвращает сеть и историю потерь."""
    torch.manual_seed(0)
    # nn.Dropout(dropout) при обучении случайно обнуляет долю dropout выходов слоя
    net = nn.Sequential(nn.Linear(X.shape[1], 256), nn.ReLU(), nn.Dropout(dropout),
                        nn.Linear(256, 256), nn.ReLU(), nn.Dropout(dropout),
                        nn.Linear(256, 1))                # ~80 тыс. параметров на 750 заёмщиков
    # AdamW — Adam с weight decay: на каждом шаге веса немного уменьшаются
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=weight_decay)
    bce = nn.BCEWithLogitsLoss()
    train_hist, val_hist = [], []
    for epoch in range(epochs):
        net.train()                                       # режим обучения: dropout включён
        opt.zero_grad()
        bce(net(X_train_t).squeeze(1), y_train_t).backward()   # один шаг на всей выборке
        opt.step()
        net.eval()                                        # режим проверки: dropout выключен
        with torch.no_grad():                             # измеряем потери после шага
            train_hist.append(bce(net(X_train_t).squeeze(1), y_train_t).item())
            val_hist.append(bce(net(X_val_t).squeeze(1), y_val_t).item())
    return net, train_hist, val_hist

for label, dropout, wd in [("без регуляризации", 0.0, 0.0), ("dropout 0.5 + weight decay 1.0", 0.5, 1.0)]:
    net, tr, va = train_big(dropout, wd)
    best = int(np.argmin(va))                             # номер эпохи с наименьшей val loss (с 0)
    with torch.no_grad():
        # логит > 0 означает вероятность > 0.5, то есть ответ «не вернёт»;
        # сравниваем с правильными ответами и берём долю совпадений
        acc = ((net(X_val_t).squeeze(1) > 0).float() == y_val_t).float().mean().item()
    print(f"{label}:")
    print(f"   train loss: эпоха 10 {tr[9]:.3f}, 50 {tr[49]:.3f}, 300 {tr[299]:.4f}")
    print(f"   val loss:   эпоха 10 {va[9]:.3f}, 50 {va[49]:.3f}, 300 {va[299]:.3f}; минимум {va[best]:.3f} на эпохе {best + 1}")
    print(f"   точность на валидации в конце: {acc:.3f}")

* Без регуляризации потеря на обучении упала до 0.001 — сеть выучила все случайности. Потеря на валидации после минимума 0.482 (эпоха 35) выросла до 1.833: это переобучение.
* С dropout и weight decay рост слабее (1.078), но к концу переобучилась и она.
* Точность почти одинаковая (0.78 и 0.76), а потеря — нет: переобученная сеть излишне уверена в своих ошибках.

Лекарство — ранняя остановка: взять веса эпохи с минимальной потерей на валидации.